# 🤖 **AI Agent with LangGraph & MLflow Tracing**
## *Production-Ready Conversational Agent Architecture*

---

### 🎯 **Project Overview**

This notebook demonstrates a **production-ready AI agent** built with **LangGraph** for orchestration, **Databricks LangChain** for LLM integration, and **MLflow** for observability. The agent can call Unity Catalog functions as tools.

### 🏗️ **Architecture Philosophy**

```mermaid
    A[User Message] --> B[Agent Node]
    B --> C{Tool Calls?}
    C -->|Yes| D[Tools Node]
    C -->|No| E[END]
    D --> B
```

### 🔥 **Key Innovation: Tool-Calling Agent**

| **Component** | **Technology** | **Why?** |
|---------------|----------------|----------|
| Agent Orchestration | **LangGraph** | ✅ State management<br>✅ Conditional routing<br>✅ Streaming support |
| LLM Integration | **ChatDatabricks** | ✅ Databricks foundation models<br>✅ Serverless compute<br>✅ Auto-scaling |
| Tool Integration | **UC Functions** | ✅ Governed access<br>✅ Unity Catalog integration<br>✅ Secure execution |
| Observability | **MLflow Tracing** | ✅ Auto-logging<br>✅ Debugging capabilities<br>✅ Performance monitoring |

### 📚 **Learning Objectives**

After completing this notebook, you will understand:
1. **LangGraph agent architecture** with tool calling
2. **StateGraph workflow** for agent orchestration
3. **Unity Catalog functions** as agent tools
4. **MLflow autologging** for LangChain agents
5. **Streaming responses** for real-time interactions
6. **Production deployment** patterns

### ⚡ **Why This Approach Works**

- **LangGraph**: Flexible state management for complex agent workflows
- **Databricks LLM**: Serverless foundation models with auto-scaling
- **UC Tools**: Governed, secure tool execution within Unity Catalog
- **MLflow**: End-to-end observability for debugging and monitoring

---

## 🚀 **Let's Begin the Journey!**

In [0]:
## 📦 **Environment Setup and Dependencies**

### 🎯 **Objective**
Import all necessary libraries for building the LangGraph agent with MLflow tracing.

### 🔧 **What We're Importing**
- **MLflow**: For experiment tracking and autologging
- **LangGraph**: For agent orchestration and state management
- **Databricks LangChain**: For LLM integration with Databricks
- **Unity Catalog Toolkit**: For tool integration

## 📦 **Importing Dependencies**

### 🎯 **Objective**
Load all required libraries for the LangGraph agent implementation.

In [0]:
# Core Imports - MLflow for tracking
import mlflow

# Core Imports - Type definitions for interfaces
from typing import Any, Generator, Optional, Sequence, Union

# Core Imports - Databricks LangChain integration
from databricks_langchain import (ChatDatabricks, UCFunctionToolkit, VectorSearchRetrieverTool)

# Core Imports - LangChain core types
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool

# Core Imports - LangGraph for agent orchestration
from langgraph.graph import END, StateGraph
from langgraph.graph.state import CompiledStateGraph

# Core Imports - LangGraph prebuilt tool node
from langgraph.prebuilt.tool_node import ToolNode

# Core Imports - MLflow LangChain integration
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode

# Core Imports - MLflow ChatAgent interface
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (ChatAgentChunk, ChatAgentMessage, ChatAgentResponse, ChatContext)

# Core Imports - Warnings configuration
import warnings
warnings.filterwarnings('ignore')

print("✅ Environment setup complete!")
print(f"📊 MLflow version: {mlflow.__version__}")

## 🔧 **Configuring the LLM Endpoint**

### 🎯 **Objective**
Configure the Databricks foundation model endpoint for the agent.

In [0]:
# Define the Databricks foundation model endpoint name
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

print(f"🔧 LLM Endpoint: {LLM_ENDPOINT_NAME}")

In [0]:
## 🤖 **Initializing the LLM**

### 🎯 **Objective**
Initialize the ChatDatabricks model with the configured endpoint.

# Initialize the ChatDatabricks model with the endpoint
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

print(f"✅ LLM initialized successfully!")
print(f"🤖 Model type: {type(llm).__name__}")

In [0]:
## 🔧 **Configuring Unity Catalog Tools**

### 🎯 **Objective**
Set up Unity Catalog functions as tools for the agent to call.

### 💡 **Educational Insight**
Unity Catalog functions provide governed, secure tool execution. The agent can call these functions to perform actions like code execution, data queries, or custom operations.

In [0]:
# Initialize tools list
tools = []

# Define Unity Catalog function names to use as tools
uc_tool_names = ["system.ai.python_exec"]

print(f"🔧 Configuring UC tools: {uc_tool_names}")

In [0]:
# Initialize UC toolkit with the specified function names
uc_toolkit = UCFunctionToolkit(function_names=uc_tool_names)

# Add UC tools to the tools list
tools.extend(uc_toolkit.tools)

print(f"✅ UC toolkit initialized!")
print(f"🔧 Total tools loaded: {len(tools)}")

In [0]:
# Define system prompt for agent behavior
system_prompt = "Responda com precisão. Se não souber a resposta diga que não sabe ao invés de inventar respostas."

print(f"💬 System prompt configured!")

In [0]:
## 🏗️ **Building the Tool-Calling Agent**

### 🎯 **Objective**
Create a LangGraph agent that can call tools based on user requests.

### 💡 **Educational Insight**
The agent uses a **StateGraph** to manage conversation state and conditional routing. When the LLM determines it needs to call a tool, the workflow routes to the tools node, executes the tool, and returns to the agent with the result.

# Define a function that creates a tool-calling agent using LangGraph
def create_tool_calling_agent(
    model: LanguageModelLike,                    # Language model to use for the agent
    tools: Union[ToolNode, Sequence[BaseTool]],  # Tools that the agent can call
    system_prompt: Optional[str] = None,         # Optional system prompt for agent behavior
) -> CompiledGraph:
    """
    Creates a LangGraph agent that can call tools.
    
    The agent uses a StateGraph to manage conversation state and conditional routing.
    When the LLM determines it needs to call a tool, the workflow routes to the tools node.
    """
    # Bind the model to the tools, enabling automatic tool calling
    model = model.bind_tools(tools)

    # Define a function to determine the next step in the workflow
    def should_continue(state: ChatAgentState):
        """
        Determines whether to continue to tools or end the conversation.
        """
        # Get current messages from the agent state
        messages = state["messages"]
        
        # Get the last message
        last_message = messages[-1]
        
        # If the last message contains tool calls, continue to tools node
        if last_message.get("tool_calls"):
            return "continue"
        
        # Otherwise, end the conversation
        else:
            return "end"

    # If a system prompt is defined, add it before user messages
    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}]
            + state["messages"]
        )
    # Otherwise, keep original messages
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])

    # Combine preprocessor with model to form execution pipeline
    model_runnable = preprocessor | model

    # Define function to call the model with processed messages
    def call_model(
        state: ChatAgentState,   # Current agent state including messages
        config: RunnableConfig,  # Additional execution configuration (optional)
    ):
        """
        Invokes the model with processed messages.
        """
        # Invoke the model with processed messages
        response = model_runnable.invoke(state, config)

        # Return response in the format expected by the workflow
        return {"messages": [response]}

    # Create a workflow using StateGraph to manage agent states and transitions
    workflow = StateGraph(ChatAgentState)

    # Add the agent (model) node to the workflow
    workflow.add_node("agent", RunnableLambda(call_model))
    
    # Add the tools node to the workflow
    workflow.add_node("tools", ChatAgentToolNode(tools))

    # Set "agent" as the entry point of the workflow
    workflow.set_entry_point("agent")

    # Add conditional edges based on should_continue function
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",  # If tool calls, go to tools node
            "end": END,           # Otherwise, end the conversation
        },
    )

    # Add edge from tools back to agent after tool execution
    workflow.add_edge("tools", "agent")

    # Compile the workflow, making it ready for use
    return workflow.compile()

print("✅ Agent creation function defined!")

In [0]:
## 🤖 **Building the LangGraph ChatAgent Wrapper**

### 🎯 **Objective**
Create a custom ChatAgent class that wraps the LangGraph agent for MLflow compatibility.

### 💡 **Educational Insight**
The `LangGraphChatAgent` class implements the MLflow `ChatAgent` interface, allowing the LangGraph agent to be used with MLflow's serving and evaluation frameworks. It provides both `predict` (complete response) and `predict_stream` (streaming response) methods.

# Define a custom ChatAgent class that wraps a LangGraph agent
class LangGraphChatAgent(ChatAgent):
    """
    Custom ChatAgent that integrates a LangGraph agent with MLflow.
    
    This class implements the MLflow ChatAgent interface, allowing the
    LangGraph agent to be used with MLflow's serving and evaluation frameworks.
    """
    
    def __init__(self, agent: CompiledStateGraph):
        """
        Initialize the class with a compiled LangGraph agent.
        
        Args:
            agent: A compiled LangGraph StateGraph
        """
        # Store the compiled agent as an instance variable
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],                # List of messages received by the agent
        context: Optional[ChatContext] = None,           # Optional additional conversation context
        custom_inputs: Optional[dict[str, Any]] = None,  # Optional custom inputs
    ) -> ChatAgentResponse:
        """
        Generate a complete response based on received messages.
        
        Args:
            messages: List of chat messages
            context: Optional conversation context
            custom_inputs: Optional custom inputs
            
        Returns:
            ChatAgentResponse with complete message history
        """
        # Convert received messages to dictionary format accepted by LangGraph
        request = {"messages": self._convert_messages_to_dict(messages)}

        # Initialize list to store messages generated during execution
        messages = []
        
        # Iterate over events generated by agent execution (streaming mode: "updates")
        for event in self.agent.stream(request, stream_mode="updates"):
            
            # Iterate over data returned by executed nodes in the workflow
            for node_data in event.values():
                
                # Add each returned message to the final result converted to ChatAgentMessage
                messages.extend(ChatAgentMessage(**msg) for msg in node_data.get("messages", []))

        # Return collected messages encapsulated in ChatAgentResponse
        return ChatAgentResponse(messages=messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],                # List of messages received by the agent
        context: Optional[ChatContext] = None,           # Optional additional conversation context
        custom_inputs: Optional[dict[str, Any]] = None,  # Optional custom inputs
    ) -> Generator[ChatAgentChunk, None, None]:
        """
        Generate streaming response (chunk by chunk).
        
        Args:
            messages: List of chat messages
            context: Optional conversation context
            custom_inputs: Optional custom inputs
            
        Yields:
            ChatAgentChunk for each message in the stream
        """
        # Convert received messages to dictionary format accepted by LangGraph
        request = {"messages": self._convert_messages_to_dict(messages)}
        
        # Iterate over events generated by agent in streaming mode ("updates")
        for event in self.agent.stream(request, stream_mode="updates"):
            
            # Iterate over data returned by executed nodes in the workflow
            for node_data in event.values():
                
                # Yield each message as a streaming chunk encapsulated in ChatAgentChunk
                yield from (
                    ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"]
                )

print("✅ LangGraphChatAgent class defined!")

In [0]:
## 📊 **Enabling MLflow Autologging**

### 🎯 **Objective**
Enable MLflow autologging for LangChain to automatically track agent executions.

### 💡 **Educational Insight**
MLflow autologging automatically logs parameters, metrics, and artifacts during agent execution. This provides full observability for debugging, performance monitoring, and reproducibility.

# Enable automatic logging of LangChain executions and parameters in MLflow
mlflow.langchain.autolog()

print("✅ MLflow autologging enabled for LangChain!")
print("📊 All agent executions will be automatically tracked")

In [0]:
## 🚀 **Creating the Agent Instance**

### 🎯 **Objective**
Instantiate the LangGraph agent with the configured LLM, tools, and system prompt.

# Create the AI agent using the LLM, available tools, and system prompt
dsa_agente_ia = create_tool_calling_agent(llm, tools, system_prompt)

print("✅ LangGraph agent created successfully!")
print(f"🤖 Agent type: {type(dsa_agente_ia).__name__}")

In [0]:
## 🤖 **Initializing the LangGraphChatAgent Wrapper**

### 🎯 **Objective**
Wrap the LangGraph agent with the MLflow-compatible ChatAgent interface.

In [0]:
# Initialize the custom LangGraphChatAgent with the created workflow
DSA_AGENTE = LangGraphChatAgent(dsa_agente_ia)

print("✅ LangGraphChatAgent initialized successfully!")
print(f"🤖 Agent wrapper type: {type(DSA_AGENTE).__name__}")

In [0]:
## 🧪 **Testing the Agent**

### 🎯 **Objective**
Test the agent with simple and streaming queries to verify functionality.

In [0]:
# Test 1: Simple predict with a greeting
print("🧪 Test 1: Simple Predict")
print("=" * 50)

response = DSA_AGENTE.predict({"messages": [{"role": "user", "content": "Oi. Testando 123!"}]})

print(f"✅ Response received!")
print(f"💬 Last message: {response.messages[-1].content}")

## 📡 **Testing Streaming Response**

### 🎯 **Objective**
Test the agent's streaming capability with a more complex query about financial concepts.

In [0]:
# Test 2: Streaming predict with a financial question
print("🧪 Test 2: Streaming Predict")
print("=" * 50)

question = "Defina o que é investimento no Tesouro Direto no Brasil"
print(f"❓ Question: {question}")
print(f"\n📡 Streaming response:\n")

for evento in DSA_AGENTE.predict_stream(
    {"messages": [{"role": "user", "content": question}]}
):
    # Print each chunk of the streaming response
    print(evento.delta.content, end="", flush=True)

print("\n\n✅ Streaming test complete!")

In [0]:
## 🎉 **Summary**

### ✅ **What We Accomplished**

1. **Environment Setup**: Imported all necessary libraries for LangGraph, MLflow, and Databricks integration
2. **LLM Configuration**: Set up ChatDatabricks with the foundation model endpoint
3. **Tool Integration**: Configured Unity Catalog functions as agent tools
4. **Agent Architecture**: Built a tool-calling agent using LangGraph StateGraph
5. **MLflow Integration**: Enabled autologging for full observability
6. **Testing**: Verified agent functionality with both predict and streaming modes

### 📊 **Next Steps**

This project will continue in the next chapter with:
- Advanced tool configurations
- Custom tool implementations
- Production deployment patterns
- Performance optimization

---

## 🚀 **Notebook Complete!**

## 📚 **Resources**

- [Databricks Agents Documentation](https://docs.databricks.com/en/generative-ai/agents.html)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [MLflow Tracing](https://mlflow.org/docs/latest/tracing/)
- [Unity Catalog Functions](https://docs.databricks.com/en/data-governance/unity-catalog/functions.html)

## 🏆 **License**

Based on Data Science Academy course material - www.datascienceacademy.com.br

---

**End of Notebook** 🎉